# Prices, inferred contracts and executable trade labels
Bloomberg TZT1/2/3 generic mappings have not been verified against individual contract identifiers. Delivery labels and rolls below are inferred using ICE expiry rules and UK bank holidays. They remain proxies. Settlement signals enter at the next settlement. Risk uses the latest 20 valid non-roll changes. Delivery hours include Amsterdam daylight saving. Coal switching inputs retain their unverified historical source.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get("TTF_PROJECT_ROOT", Path.cwd().parent))
RAW = ROOT / "1. Raw Data"
OUT = ROOT / "2. Data Processing"
px = pd.read_csv(RAW / "prices.csv", parse_dates=["date"])
px = px.sort_values("date").reset_index(drop=True)
px["Coal_M1"] = px["Coal_M1"].ffill(limit=3)
px = px.dropna(subset=["TTF_M1", "TTF_M2", "TTF_M3"]).reset_index(drop=True)
px = px.rename(columns={"TTF_M1": "m1", "TTF_M2": "m2", "TTF_M3": "m3"})
for spread, a, b in [("s12", "m1", "m2"), ("s23", "m2", "m3"), ("s13", "m1", "m3")]:
    px[spread] = px[a] - px[b]
    px[f"{spread}_pct"] = px[spread] / px[b] * 100

In [2]:
from dateutil.easter import easter

# England/Wales UK business-day calendar, including historical one-off holidays.
# ICE two-UK-business-day expiry rule; this does not verify Bloomberg generic rolls.
holidays = set()
for year in range(2010, max(2028, px["date"].max().year + 3)):
    jan1 = pd.Timestamp(year, 1, 1)
    holidays.add(jan1 if jan1.weekday() < 5 else jan1 + pd.Timedelta(days=7 - jan1.weekday()))
    sun = pd.Timestamp(easter(year))
    holidays.update([sun - pd.Timedelta(days=2), sun + pd.Timedelta(days=1)])
    may = pd.Timestamp(year, 5, 1)
    early = may + pd.Timedelta(days=(-may.weekday()) % 7)
    holidays.add(pd.Timestamp("2020-05-08") if year == 2020 else early)
    last_may = pd.Timestamp(year, 5, 31)
    spring = last_may - pd.Timedelta(days=last_may.weekday())
    if year == 2012:
        holidays.update(pd.to_datetime(["2012-06-04", "2012-06-05"]))
    elif year == 2022:
        holidays.update(pd.to_datetime(["2022-06-02", "2022-06-03", "2022-09-19"]))
    else:
        holidays.add(spring)
    aug = pd.Timestamp(year, 8, 31)
    holidays.add(aug - pd.Timedelta(days=aug.weekday()))
    occupied = set()
    for day in [25, 26]:
        observed = pd.Timestamp(year, 12, day)
        while observed.weekday() >= 5 or observed in occupied:
            observed += pd.Timedelta(days=1)
        occupied.add(observed)
    holidays.update(occupied)
holidays.update(pd.to_datetime(["2011-04-29", "2023-05-08"]))
uk = pd.offsets.CustomBusinessDay(holidays=sorted(holidays))
months = pd.date_range(px["date"].min().replace(day=1), px["date"].max() + pd.offsets.MonthBegin(4), freq="MS")
exp = pd.DataFrame({"delivery": months, "expiry": [d - 2 * uk for d in months]})
px = pd.merge_asof(px, exp.sort_values("expiry"), left_on="date", right_on="expiry", direction="forward")
assert px["delivery"].notna().all()
px["roll_day"] = px["delivery"].ne(px["delivery"].shift()).astype(int)
px.loc[0, "roll_day"] = 0
bus_calendar = np.busdaycalendar(holidays=np.array(sorted(holidays), dtype="datetime64[D]"))
px["dte"] = np.busday_count(px["date"].values.astype("datetime64[D]"), px["expiry"].values.astype("datetime64[D]"), busdaycal=bus_calendar)
px["contracts_source_verified"] = False
for leg in [1, 2, 3]:
    delivery = px["delivery"] + pd.DateOffset(months=leg - 1)
    px[f"m{leg}_contract"] = delivery.dt.strftime("%Y-%m")
    starts = pd.DatetimeIndex(delivery + pd.Timedelta(hours=6)).tz_localize("Europe/Amsterdam")
    ends = pd.DatetimeIndex(delivery + pd.DateOffset(months=1) + pd.Timedelta(hours=6)).tz_localize("Europe/Amsterdam")
    px[f"m{leg}_hours"] = ((ends.tz_convert("UTC") - starts.tz_convert("UTC")) / pd.Timedelta(hours=1)).astype(int)
for month, name in enumerate(["d_feb", "d_mar", "d_apr", "d_may", "d_jun", "d_jul", "d_aug", "d_sep", "d_oct", "d_nov", "d_dec"], start=2):
    px[name] = px["delivery"].dt.month.eq(month).astype(int)
px["inj_season"] = px["date"].dt.month.between(4, 9).astype(int)
# Ex-post explanatory flag: excluded from predictive baseline feature lists.
px["crisis"] = px["date"].between("2021-07-01", "2023-03-31").astype(int)

In [3]:
# Trailing 20 valid non-roll observations, carried until next valid update.
# A generic-series risk proxy, not individual-contract realised volatility.
for price, vol in [("m1", "vol_m1"), ("m2", "vol_m2"), ("m3", "vol_m3")]:
    ret = np.log(px[price] / px[price].shift(1)).mask(px["roll_day"].eq(1))
    px[vol] = ret.dropna().rolling(20, min_periods=20).std().reindex(px.index).ffill() * np.sqrt(252) * 100
for spread in ["s12", "s23", "s13"]:
    change = px[spread].diff().mask(px["roll_day"].eq(1))
    px[f"sd_d{spread}"] = change.dropna().rolling(20, min_periods=20).std().reindex(px.index).ffill()
    clean5 = px["roll_day"].rolling(5, min_periods=5).sum().eq(0)
    px[f"mom_d{spread}_5"] = px[spread].diff(5).where(clean5)
px["risk_estimate_proxy"] = True
roll_ahead = px["roll_day"].rolling(5).sum().shift(-5)
for spread in ["s12", "s23", "s13"]:
    px[f"d{spread}_fwd5"] = (px[spread].shift(-5) - px[spread]).mask(roll_ahead.gt(0))

In [4]:
# Forecast at signal t; execution next settlement, then hold h sessions.
# No trade label survives a change of inferred delivery in any leg.
contracts = px[["m1_contract", "m2_contract", "m3_contract"]]
for h in [1, 5, 10]:
    px[f"entry_date{h}"] = px["date"].shift(-1)
    px[f"exit_date{h}"] = px["date"].shift(-(1 + h))
    same_entry = contracts.eq(contracts.shift(-1)).all(axis=1)
    same_exit = contracts.eq(contracts.shift(-(1 + h))).all(axis=1)
    valid = same_entry & same_exit & px[f"exit_date{h}"].notna()
    px[f"trade_valid{h}"] = valid
    for spread in ["s12", "s23", "s13"]:
        px[f"d{spread}_trade{h}"] = (px[spread].shift(-(1 + h)) - px[spread].shift(-1)).where(valid)

In [5]:
coal_eur = px["Coal_M1"] / px["EUR/USD"] / 6.978
px["ttf_switch"] = (0.52 / 0.38) * (coal_eur + 0.341 * px["EUA_M1"]) - 0.202 * px["EUA_M1"]
px["switch_gap"] = px["m1"] - px["ttf_switch"]
px["coal_source_verified"] = False
px["switch_source_verified"] = False
px = px.drop(columns=["M1_M2_spread", "M1_M3_spread", "M2_M3_spread", "Coal_M1", "EUR/USD", "EUA_M1"], errors="ignore")
assert np.allclose(px["s13"], px["s12"] + px["s23"])
assert px["dte"].ge(0).all()
px.to_csv(OUT / "prices_processed.csv", index=False)